# 데이터 품질 점검


- 중복 id 확인
- answers["text"]가 빈 문자열인 경우
- context가 비어 있거나 너무 짧은 경우
- answer_start가 음수인지
- 잘못된 annotation(check: substring mismatch)
- answer_start + answer_text 길이가 context 길이를 초과하는지
- 정답이 중복 등장하는 경우 확인
- 공백/개행 문자로 인한 mismatch 검사
- answer_start 오프셋 오류 확인 (±1 위치)
- Context 길이 이상치 확인 (IQR 방식)
- Question 길이 이상치 확인
- 중복된 (question, context) 쌍 확인
- Title과 Context 불일치 확인
- HTML 태그/Markup 포함 여부 확인
- Answer Text 앞뒤 공백 확인


In [14]:
# 라이브러리 임포트
from datasets import load_from_disk
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm

print("라이브러리 임포트 완료")


라이브러리 임포트 완료


In [15]:
# 데이터 로드
print("데이터 로드 중...")
train_dataset = load_from_disk("../../data/train_dataset")

train_df = pd.DataFrame(train_dataset["train"])
val_df = pd.DataFrame(train_dataset["validation"])

print(f"Train 데이터: {len(train_df)}개")
print(f"Validation 데이터: {len(val_df)}개")
print(f"\n컬럼: {train_df.columns.tolist()}")
print(f"\n샘플 데이터:")
train_df.head(2)


데이터 로드 중...
Train 데이터: 3952개
Validation 데이터: 240개

컬럼: ['title', 'context', 'question', 'id', 'answers', 'document_id', '__index_level_0__']

샘플 데이터:


,title,context,question,id,answers,document_id,__index_level_0__
0,미국 상원,미국 상의원 또는 미국 상원(United States Senate)은 양원제인 미국...,대통령을 포함한 미국의 행정부 견제권을 갖는 국가 기관은?,mrc-1-000067,"{'answer_start': [235], 'text': ['하원']}",18293,42
1,인사조직관리,'근대적 경영학' 또는 '고전적 경영학'에서 현대적 경영학으로 전환되는 시기는 19...,현대적 인사조직관리의 시발점이 된 책은?,mrc-0-004397,"{'answer_start': [212], 'text': ['《경영의 실제》']}",51638,2873


## 1. 중복 id 확인


In [16]:
# Train 데이터 중복 id 확인
train_duplicate_ids = train_df[train_df.duplicated(subset=['id'], keep=False)]
val_duplicate_ids = val_df[val_df.duplicated(subset=['id'], keep=False)]

print("=" * 50)
print("1. 중복 ID 확인")
print("=" * 50)
print(f"Train 데이터 중복 ID 개수: {len(train_duplicate_ids)}")
print(f"Validation 데이터 중복 ID 개수: {len(val_duplicate_ids)}")

if len(train_duplicate_ids) > 0:
    print(f"\nTrain 중복 ID 목록:")
    print(train_duplicate_ids[['id', 'question']].head(10))
    
if len(val_duplicate_ids) > 0:
    print(f"\nValidation 중복 ID 목록:")
    print(val_duplicate_ids[['id', 'question']].head(10))

# 중복 ID가 있는 경우 상세 정보
if len(train_duplicate_ids) > 0:
    duplicate_id_counts = train_df['id'].value_counts()
    duplicate_id_counts = duplicate_id_counts[duplicate_id_counts > 1]
    print(f"\n중복된 ID별 개수:")
    print(duplicate_id_counts.head(10))


1. 중복 ID 확인
Train 데이터 중복 ID 개수: 0
Validation 데이터 중복 ID 개수: 0


## 2. answers["text"]가 빈 문자열인 경우


In [17]:
# answers["text"]가 빈 문자열인 경우 확인
def check_empty_answers(df, dataset_name):
    empty_answer_indices = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        if isinstance(answers, dict) and 'text' in answers:
            texts = answers['text']
            if isinstance(texts, list):
                # 빈 문자열이 있는지 확인
                if any(text == '' or text is None for text in texts):
                    empty_answer_indices.append(idx)
    
    return empty_answer_indices

train_empty_answers = check_empty_answers(train_df, "Train")
val_empty_answers = check_empty_answers(val_df, "Validation")

print("=" * 50)
print("2. 빈 문자열 answer 확인")
print("=" * 50)
print(f"Train 데이터 빈 answer 개수: {len(train_empty_answers)}")
print(f"Validation 데이터 빈 answer 개수: {len(val_empty_answers)}")

if len(train_empty_answers) > 0:
    print(f"\nTrain 빈 answer 샘플:")
    print(train_df.loc[train_empty_answers, ['id', 'question', 'answers']].head(10))
    
if len(val_empty_answers) > 0:
    print(f"\nValidation 빈 answer 샘플:")
    print(val_df.loc[val_empty_answers, ['id', 'question', 'answers']].head(10))


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 12982.28it/s]

2. 빈 문자열 answer 확인
Train 데이터 빈 answer 개수: 0
Validation 데이터 빈 answer 개수: 0


## 3. context가 비어 있거나 너무 짧은 경우


In [18]:
# context 길이 분석
train_df['context_length'] = train_df['context'].apply(len)
val_df['context_length'] = val_df['context'].apply(len)

# 빈 context 또는 너무 짧은 context 확인 (예: 10자 이하)
MIN_CONTEXT_LENGTH = 10

train_empty_context = train_df[train_df['context_length'] == 0]
train_short_context = train_df[train_df['context_length'] < MIN_CONTEXT_LENGTH]

val_empty_context = val_df[val_df['context_length'] == 0]
val_short_context = val_df[val_df['context_length'] < MIN_CONTEXT_LENGTH]

print("=" * 50)
print("3. Context 길이 확인")
print("=" * 50)
print(f"Train 데이터:")
print(f"  - 빈 context: {len(train_empty_context)}개")
print(f"  - {MIN_CONTEXT_LENGTH}자 미만 context: {len(train_short_context)}개")
print(f"  - 평균 context 길이: {train_df['context_length'].mean():.2f}자")
print(f"  - 최소 context 길이: {train_df['context_length'].min()}자")
print(f"  - 최대 context 길이: {train_df['context_length'].max()}자")

print(f"\nValidation 데이터:")
print(f"  - 빈 context: {len(val_empty_context)}개")
print(f"  - {MIN_CONTEXT_LENGTH}자 미만 context: {len(val_short_context)}개")
print(f"  - 평균 context 길이: {val_df['context_length'].mean():.2f}자")
print(f"  - 최소 context 길이: {val_df['context_length'].min()}자")
print(f"  - 최대 context 길이: {val_df['context_length'].max()}자")

if len(train_short_context) > 0:
    print(f"\nTrain 짧은 context 샘플:")
    print(train_short_context[['id', 'context_length', 'context']].head(10))


3. Context 길이 확인
Train 데이터:
  - 빈 context: 0개
  - 10자 미만 context: 0개
  - 평균 context 길이: 920.22자
  - 최소 context 길이: 512자
  - 최대 context 길이: 2059자

Validation 데이터:
  - 빈 context: 0개
  - 10자 미만 context: 0개
  - 평균 context 길이: 916.73자
  - 최소 context 길이: 517자
  - 최대 context 길이: 2064자


## 4. answer_start가 음수인지 확인


In [19]:
# answer_start가 음수이거나 context 길이를 넘어가는 경우 확인
def check_answer_start(df, dataset_name):
    negative_start_indices = []
    out_of_range_indices = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        context_len = len(context)
        
        if isinstance(answers, dict) and 'answer_start' in answers:
            starts = answers['answer_start']
            if isinstance(starts, list):
                for start in starts:
                    if start < 0:
                        negative_start_indices.append(idx)
                    elif start >= context_len:
                        out_of_range_indices.append(idx)
    
    return negative_start_indices, out_of_range_indices

train_neg_start, train_out_range = check_answer_start(train_df, "Train")
val_neg_start, val_out_range = check_answer_start(val_df, "Validation")

print("=" * 50)
print("4. answer_start 범위 확인")
print("=" * 50)
print(f"Train 데이터:")
print(f"  - 음수 answer_start: {len(train_neg_start)}개")
print(f"  - context 길이 초과 answer_start: {len(train_out_range)}개")

print(f"\nValidation 데이터:")
print(f"  - 음수 answer_start: {len(val_neg_start)}개")
print(f"  - context 길이 초과 answer_start: {len(val_out_range)}개")

if len(train_neg_start) > 0:
    print(f"\nTrain 음수 answer_start 샘플:")
    problem_df = train_df.loc[train_neg_start, ['id', 'answers', 'context_length']]
    print(problem_df.head(10))
    
if len(train_out_range) > 0:
    print(f"\nTrain 범위 초과 answer_start 샘플:")
    problem_df = train_df.loc[train_out_range, ['id', 'answers', 'context_length']]
    for idx in train_out_range[:5]:
        row = train_df.loc[idx]
        print(f"\nID: {row['id']}")
        print(f"Context 길이: {row['context_length']}")
        print(f"Answer start: {row['answers']['answer_start']}")
        print(f"Context: {row['context'][:100]}...")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 10416.53it/s]

4. answer_start 범위 확인
Train 데이터:
  - 음수 answer_start: 0개
  - context 길이 초과 answer_start: 0개

Validation 데이터:
  - 음수 answer_start: 0개
  - context 길이 초과 answer_start: 0개


## 5. 잘못된 annotation 확인 (substring mismatch)


In [20]:
# answer_start 위치의 텍스트가 실제 answer text와 일치하는지 확인
def check_annotation_mismatch(df, dataset_name):
    mismatch_indices = []
    mismatch_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    
                    # answer_start 위치에서 추출한 텍스트
                    extracted_text = context[start:start+len(text)]
                    
                    # 정확히 일치하는지 확인
                    if extracted_text != text:
                        mismatch_indices.append(idx)
                        mismatch_details.append({
                            'id': row['id'],
                            'expected': text,
                            'extracted': extracted_text,
                            'start': start,
                            'context_snippet': context[max(0, start-20):start+len(text)+20]
                        })
                        break  # 하나라도 불일치하면 해당 인덱스 추가
    
    return mismatch_indices, mismatch_details

train_mismatch_idx, train_mismatch_details = check_annotation_mismatch(train_df, "Train")
val_mismatch_idx, val_mismatch_details = check_annotation_mismatch(val_df, "Validation")

print("=" * 50)
print("5. Annotation 불일치 확인")
print("=" * 50)
print(f"Train 데이터 annotation 불일치: {len(train_mismatch_idx)}개")
print(f"Validation 데이터 annotation 불일치: {len(val_mismatch_idx)}개")

if len(train_mismatch_details) > 0:
    print(f"\nTrain annotation 불일치 샘플 (최대 10개):")
    for i, detail in enumerate(train_mismatch_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer start: {detail['start']}")
        print(f"    예상 텍스트: '{detail['expected']}'")
        print(f"    추출된 텍스트: '{detail['extracted']}'")
        print(f"    Context 주변: ...{detail['context_snippet']}...")
        
if len(val_mismatch_details) > 0:
    print(f"\nValidation annotation 불일치 샘플 (최대 10개):")
    for i, detail in enumerate(val_mismatch_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer start: {detail['start']}")
        print(f"    예상 텍스트: '{detail['expected']}'")
        print(f"    추출된 텍스트: '{detail['extracted']}'")
        print(f"    Context 주변: ...{detail['context_snippet']}...")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 11382.10it/s]

5. Annotation 불일치 확인
Train 데이터 annotation 불일치: 0개
Validation 데이터 annotation 불일치: 0개


## 6. answer_start + answer_text 길이가 context 길이를 초과하는지


In [21]:
# answer_start + answer_text 길이가 context 길이를 초과하는지 확인
def check_answer_span_out_of_range(df, dataset_name):
    out_of_range_indices = []
    out_of_range_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        context_len = len(context)
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0:
                        continue
                    answer_end = start + len(text)
                    if answer_end > context_len:
                        out_of_range_indices.append(idx)
                        out_of_range_details.append({
                            'id': row['id'],
                            'answer_start': start,
                            'answer_text': text,
                            'answer_text_len': len(text),
                            'answer_end': answer_end,
                            'context_len': context_len,
                            'overflow': answer_end - context_len
                        })
                        break
    
    return out_of_range_indices, out_of_range_details

train_span_out, train_span_details = check_answer_span_out_of_range(train_df, "Train")
val_span_out, val_span_details = check_answer_span_out_of_range(val_df, "Validation")

print("=" * 50)
print("6. Answer Span 범위 초과 확인")
print("=" * 50)
print(f"Train 데이터 span 범위 초과: {len(train_span_out)}개")
print(f"Validation 데이터 span 범위 초과: {len(val_span_out)}개")

if len(train_span_details) > 0:
    print(f"\nTrain span 범위 초과 샘플 (최대 10개):")
    for i, detail in enumerate(train_span_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer start: {detail['answer_start']}")
        print(f"    Answer text: '{detail['answer_text']}' (길이: {detail['answer_text_len']})")
        print(f"    Answer end: {detail['answer_end']}")
        print(f"    Context 길이: {detail['context_len']}")
        print(f"    초과량: {detail['overflow']}자")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 11594.62it/s]

6. Answer Span 범위 초과 확인
Train 데이터 span 범위 초과: 0개
Validation 데이터 span 범위 초과: 0개


## 7. 정답이 중복 등장하는 경우 확인


In [22]:
# 정답이 context 안에 여러 번 등장하는 경우 확인
def check_duplicate_answer_occurrences(df, dataset_name):
    duplicate_occurrence_indices = []
    duplicate_occurrence_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    
                    # context에서 answer_text가 등장하는 모든 위치 찾기
                    occurrences = []
                    search_start = 0
                    while True:
                        pos = context.find(text, search_start)
                        if pos == -1:
                            break
                        occurrences.append(pos)
                        search_start = pos + 1
                    
                    # 여러 번 등장하는 경우
                    if len(occurrences) > 1:
                        duplicate_occurrence_indices.append(idx)
                        duplicate_occurrence_details.append({
                            'id': row['id'],
                            'answer_text': text,
                            'answer_start': start,
                            'all_occurrences': occurrences,
                            'occurrence_count': len(occurrences)
                        })
                        break
    
    return duplicate_occurrence_indices, duplicate_occurrence_details

train_dup_ans, train_dup_details = check_duplicate_answer_occurrences(train_df, "Train")
val_dup_ans, val_dup_details = check_duplicate_answer_occurrences(val_df, "Validation")

print("=" * 50)
print("7. 정답 중복 등장 확인")
print("=" * 50)
print(f"Train 데이터 정답 중복 등장: {len(train_dup_ans)}개")
print(f"Validation 데이터 정답 중복 등장: {len(val_dup_ans)}개")

if len(train_dup_details) > 0:
    print(f"\nTrain 정답 중복 등장 샘플 (최대 10개):")
    for i, detail in enumerate(train_dup_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer text: '{detail['answer_text']}'")
        print(f"    Annotated start: {detail['answer_start']}")
        print(f"    모든 등장 위치: {detail['all_occurrences']}")
        print(f"    등장 횟수: {detail['occurrence_count']}회")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 9188.72it/s]

7. 정답 중복 등장 확인
Train 데이터 정답 중복 등장: 1312개
Validation 데이터 정답 중복 등장: 78개

Train 정답 중복 등장 샘플 (최대 10개):

[1] ID: mrc-1-000067
    Answer text: '하원'
    Annotated start: 235
    모든 등장 위치: [183, 235, 436, 449, 498, 543, 594, 677, 781, 800]
    등장 횟수: 10회

[2] ID: mrc-1-000362
    Answer text: '백성'
    Annotated start: 510
    모든 등장 위치: [366, 510, 645, 666, 813]
    등장 횟수: 5회

[3] ID: mrc-0-005265
    Answer text: '드래곤'
    Annotated start: 91
    모든 등장 위치: [91, 105, 161, 186, 437]
    등장 횟수: 5회

[4] ID: mrc-0-003839
    Answer text: '왕대마을'
    Annotated start: 861
    모든 등장 위치: [861, 929, 1031]
    등장 횟수: 3회

[5] ID: mrc-1-001008
    Answer text: '예수'
    Annotated start: 497
    모든 등장 위치: [428, 497]
    등장 횟수: 2회

[6] ID: mrc-0-002011
    Answer text: '1916년'
    Annotated start: 414
    모든 등장 위치: [178, 231, 414, 639]
    등장 횟수: 4회

[7] ID: mrc-0-000424
    Answer text: '레드삭스'
    Annotated start: 13
    모든 등장 위치: [13, 207]
    등장 횟수: 2회

[8] ID: mrc-1-000027
    Answer text: '다산 정약용'
    An

## 8. 공백/개행 문자로 인한 mismatch 검사


In [23]:
# 공백/개행 문자로 인한 mismatch 검사
import unicodedata
import re

def normalize_whitespace(text):
    """공백 문자 정규화"""
    # 여러 공백을 하나로
    text = re.sub(r'\s+', ' ', text)
    # 앞뒤 공백 제거
    text = text.strip()
    return text

def check_whitespace_mismatch(df, dataset_name):
    whitespace_mismatch_indices = []
    whitespace_mismatch_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    
                    # 원본 추출
                    extracted = context[start:start+len(text)]
                    
                    # 정규화된 버전 비교
                    normalized_extracted = normalize_whitespace(extracted)
                    normalized_text = normalize_whitespace(text)
                    
                    # 원본은 다르지만 정규화 후 같으면 whitespace 문제
                    if extracted != text and normalized_extracted == normalized_text:
                        whitespace_mismatch_indices.append(idx)
                        whitespace_mismatch_details.append({
                            'id': row['id'],
                            'answer_text': text,
                            'extracted': extracted,
                            'answer_start': start,
                            'has_tab': '\t' in extracted or '\t' in text,
                            'has_newline': '\n' in extracted or '\n' in text,
                            'has_multiple_spaces': '  ' in extracted or '  ' in text
                        })
                        break
    
    return whitespace_mismatch_indices, whitespace_mismatch_details

train_ws_mismatch, train_ws_details = check_whitespace_mismatch(train_df, "Train")
val_ws_mismatch, val_ws_details = check_whitespace_mismatch(val_df, "Validation")

print("=" * 50)
print("8. 공백/개행 문자 Mismatch 확인")
print("=" * 50)
print(f"Train 데이터 whitespace mismatch: {len(train_ws_mismatch)}개")
print(f"Validation 데이터 whitespace mismatch: {len(val_ws_mismatch)}개")

if len(train_ws_details) > 0:
    print(f"\nTrain whitespace mismatch 샘플 (최대 10개):")
    for i, detail in enumerate(train_ws_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer text: {repr(detail['answer_text'])}")
        print(f"    Extracted: {repr(detail['extracted'])}")
        print(f"    Tab 포함: {detail['has_tab']}, Newline 포함: {detail['has_newline']}, 다중 공백: {detail['has_multiple_spaces']}")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 9742.58it/s]

8. 공백/개행 문자 Mismatch 확인
Train 데이터 whitespace mismatch: 0개
Validation 데이터 whitespace mismatch: 0개


## 9. answer_start 오프셋 오류 확인 (±1 위치)


In [24]:
# answer_start가 ±1 위치에 실제로 있는지 확인 (주석 어긋남 오류)
def check_offset_mismatch(df, dataset_name):
    offset_mismatch_indices = []
    offset_mismatch_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    
                    # 정확한 위치
                    exact_match = context[start:start+len(text)] == text
                    
                    # ±1 위치 확인
                    match_at_minus1 = False
                    match_at_plus1 = False
                    
                    if start > 0:
                        match_at_minus1 = context[start-1:start-1+len(text)] == text
                    if start + len(text) < len(context):
                        match_at_plus1 = context[start+1:start+1+len(text)] == text
                    
                    # 정확한 위치는 아니지만 ±1에 있으면 오프셋 오류
                    if not exact_match and (match_at_minus1 or match_at_plus1):
                        offset_mismatch_indices.append(idx)
                        offset_mismatch_details.append({
                            'id': row['id'],
                            'answer_text': text,
                            'answer_start': start,
                            'exact_match': exact_match,
                            'match_at_minus1': match_at_minus1,
                            'match_at_plus1': match_at_plus1
                        })
                        break
    
    return offset_mismatch_indices, offset_mismatch_details

train_offset_mismatch, train_offset_details = check_offset_mismatch(train_df, "Train")
val_offset_mismatch, val_offset_details = check_offset_mismatch(val_df, "Validation")

print("=" * 50)
print("9. Answer Start 오프셋 오류 확인")
print("=" * 50)
print(f"Train 데이터 오프셋 오류: {len(train_offset_mismatch)}개")
print(f"Validation 데이터 오프셋 오류: {len(val_offset_mismatch)}개")

if len(train_offset_details) > 0:
    print(f"\nTrain 오프셋 오류 샘플 (최대 10개):")
    for i, detail in enumerate(train_offset_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer text: '{detail['answer_text']}'")
        print(f"    Answer start: {detail['answer_start']}")
        print(f"    정확한 위치 일치: {detail['exact_match']}")
        print(f"    -1 위치 일치: {detail['match_at_minus1']}")
        print(f"    +1 위치 일치: {detail['match_at_plus1']}")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 10587.78it/s]

9. Answer Start 오프셋 오류 확인
Train 데이터 오프셋 오류: 0개
Validation 데이터 오프셋 오류: 0개


## 10. Context 길이 이상치 확인 (IQR 방식)


In [25]:
# Context 길이 이상치 확인 (IQR 방식)
def detect_context_outliers(df, dataset_name):
    context_lengths = df['context_length'].values
    
    Q1 = np.percentile(context_lengths, 25)
    Q3 = np.percentile(context_lengths, 75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df['context_length'] < lower_bound) | (df['context_length'] > upper_bound)]
    
    return outliers, lower_bound, upper_bound, Q1, Q3, IQR

train_outliers, train_lb, train_ub, train_q1, train_q3, train_iqr = detect_context_outliers(train_df, "Train")
val_outliers, val_lb, val_ub, val_q1, val_q3, val_iqr = detect_context_outliers(val_df, "Validation")

print("=" * 50)
print("10. Context 길이 이상치 확인 (IQR)")
print("=" * 50)
print(f"\nTrain 데이터:")
print(f"  - Q1: {train_q1:.0f}자, Q3: {train_q3:.0f}자, IQR: {train_iqr:.0f}자")
print(f"  - 정상 범위: {train_lb:.0f} ~ {train_ub:.0f}자")
print(f"  - 이상치 개수: {len(train_outliers)}개 ({len(train_outliers)/len(train_df)*100:.2f}%)")
if len(train_outliers) > 0:
    print(f"  - 최소 이상치: {train_outliers['context_length'].min()}자")
    print(f"  - 최대 이상치: {train_outliers['context_length'].max()}자")

print(f"\nValidation 데이터:")
print(f"  - Q1: {val_q1:.0f}자, Q3: {val_q3:.0f}자, IQR: {val_iqr:.0f}자")
print(f"  - 정상 범위: {val_lb:.0f} ~ {val_ub:.0f}자")
print(f"  - 이상치 개수: {len(val_outliers)}개 ({len(val_outliers)/len(val_df)*100:.2f}%)")
if len(val_outliers) > 0:
    print(f"  - 최소 이상치: {val_outliers['context_length'].min()}자")
    print(f"  - 최대 이상치: {val_outliers['context_length'].max()}자")

if len(train_outliers) > 0:
    print(f"\nTrain 이상치 샘플 (최대 5개):")
    print(train_outliers[['id', 'context_length', 'question']].head(5))


10. Context 길이 이상치 확인 (IQR)

Train 데이터:
  - Q1: 645자, Q3: 1099자, IQR: 454자
  - 정상 범위: -36 ~ 1781자
  - 이상치 개수: 155개 (3.92%)
  - 최소 이상치: 1781자
  - 최대 이상치: 2059자

Validation 데이터:
  - Q1: 617자, Q3: 1107자, IQR: 490자
  - 정상 범위: -119 ~ 1843자
  - 이상치 개수: 7개 (2.92%)
  - 최소 이상치: 1857자
  - 최대 이상치: 2064자

Train 이상치 샘플 (최대 5개):
              id  context_length                             question
9   mrc-0-003839            1826  고려 공민왕이 처가 식구들과 아내와 함께 피신처로 삼은 마을은?
35  mrc-1-000571            1944                      브루노 라이헨바흐의 직업은?
37  mrc-0-004459            1882             항공기에 기름을 바르는 것은 누구의 일인가?
51  mrc-0-005019            1811    도비가 아메리칸 리그에 데뷔할 수 있게 그를 발탁한 인물은?
81  mrc-0-000655            1908              조지가 보트를 대여할 때 사용했던 이름은?


## 11. Question 길이 이상치 확인


In [26]:
# Question 길이 분석 및 이상치 확인
train_df['question_length'] = train_df['question'].apply(len)
val_df['question_length'] = val_df['question'].apply(len)

# 비정상적으로 긴 질문 확인 (예: 500자 이상)
QUESTION_LENGTH_THRESHOLD = 500

train_long_questions = train_df[train_df['question_length'] > QUESTION_LENGTH_THRESHOLD]
val_long_questions = val_df[val_df['question_length'] > QUESTION_LENGTH_THRESHOLD]

print("=" * 50)
print("11. Question 길이 이상치 확인")
print("=" * 50)
print(f"Train 데이터:")
print(f"  - 평균 질문 길이: {train_df['question_length'].mean():.2f}자")
print(f"  - 중앙값 질문 길이: {train_df['question_length'].median():.0f}자")
print(f"  - 최대 질문 길이: {train_df['question_length'].max()}자")
print(f"  - {QUESTION_LENGTH_THRESHOLD}자 초과 질문: {len(train_long_questions)}개")

print(f"\nValidation 데이터:")
print(f"  - 평균 질문 길이: {val_df['question_length'].mean():.2f}자")
print(f"  - 중앙값 질문 길이: {val_df['question_length'].median():.0f}자")
print(f"  - 최대 질문 길이: {val_df['question_length'].max()}자")
print(f"  - {QUESTION_LENGTH_THRESHOLD}자 초과 질문: {len(val_long_questions)}개")

if len(train_long_questions) > 0:
    print(f"\nTrain 긴 질문 샘플 (최대 5개):")
    for idx in train_long_questions.index[:5]:
        row = train_df.loc[idx]
        print(f"\nID: {row['id']}")
        print(f"질문 길이: {row['question_length']}자")
        print(f"질문: {row['question'][:200]}...")


11. Question 길이 이상치 확인
Train 데이터:
  - 평균 질문 길이: 29.32자
  - 중앙값 질문 길이: 29자
  - 최대 질문 길이: 78자
  - 500자 초과 질문: 0개

Validation 데이터:
  - 평균 질문 길이: 29.20자
  - 중앙값 질문 길이: 29자
  - 최대 질문 길이: 59자
  - 500자 초과 질문: 0개


## 12. 중복된 (question, context) 쌍 확인


In [27]:
# 중복된 (question, context) 쌍 확인
def check_duplicate_question_context_pairs(df, dataset_name):
    # question과 context 조합으로 중복 확인
    df_with_pair = df.copy()
    df_with_pair['question_context_pair'] = df_with_pair['question'] + '|||' + df_with_pair['context']
    
    duplicates = df_with_pair[df_with_pair.duplicated(subset=['question_context_pair'], keep=False)]
    
    return duplicates

train_dup_pairs = check_duplicate_question_context_pairs(train_df, "Train")
val_dup_pairs = check_duplicate_question_context_pairs(val_df, "Validation")

print("=" * 50)
print("12. 중복된 (Question, Context) 쌍 확인")
print("=" * 50)
print(f"Train 데이터 중복 쌍: {len(train_dup_pairs)}개")
print(f"Validation 데이터 중복 쌍: {len(val_dup_pairs)}개")

if len(train_dup_pairs) > 0:
    print(f"\nTrain 중복 쌍 샘플 (최대 10개):")
    print(train_dup_pairs[['id', 'question', 'context_length']].head(10))
    
    # 중복 그룹별 개수
    pair_counts = train_dup_pairs.groupby('question_context_pair').size()
    print(f"\n중복 그룹별 개수 (상위 5개):")
    print(pair_counts.sort_values(ascending=False).head(5))


12. 중복된 (Question, Context) 쌍 확인
Train 데이터 중복 쌍: 0개
Validation 데이터 중복 쌍: 0개


## 13. Title과 Context 불일치 확인


In [28]:
# Title과 Context의 관련성 확인 (간단한 키워드 매칭)
def check_title_context_mismatch(df, dataset_name):
    mismatch_indices = []
    mismatch_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        title = str(row['title']) if 'title' in row else ''
        context = str(row['context'])
        
        if not title or not context:
            continue
        
        # Title의 주요 키워드가 context에 포함되는지 확인
        # 간단한 방법: title의 단어들이 context에 나타나는지
        title_words = set(title.split())
        context_words = set(context.split())
        
        # 공통 단어 비율
        if len(title_words) > 0:
            common_words = title_words & context_words
            overlap_ratio = len(common_words) / len(title_words)
            
            # 공통 단어가 너무 적으면 불일치 가능성
            if overlap_ratio < 0.1 and len(title_words) > 2:  # threshold 조정 가능
                mismatch_indices.append(idx)
                mismatch_details.append({
                    'id': row['id'],
                    'title': title,
                    'title_words': len(title_words),
                    'common_words': len(common_words),
                    'overlap_ratio': overlap_ratio,
                    'context_preview': context[:200]
                })
    
    return mismatch_indices, mismatch_details

train_title_mismatch, train_title_details = check_title_context_mismatch(train_df, "Train")
val_title_mismatch, val_title_details = check_title_context_mismatch(val_df, "Validation")

print("=" * 50)
print("13. Title과 Context 불일치 확인")
print("=" * 50)
print(f"Train 데이터 title-context 불일치 의심: {len(train_title_mismatch)}개")
print(f"Validation 데이터 title-context 불일치 의심: {len(val_title_mismatch)}개")

if len(train_title_details) > 0:
    print(f"\nTrain 불일치 의심 샘플 (최대 10개):")
    for i, detail in enumerate(train_title_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Title: '{detail['title']}'")
        print(f"    공통 단어 비율: {detail['overlap_ratio']:.2%}")
        print(f"    Context 미리보기: {detail['context_preview']}...")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 6769.01it/s]

13. Title과 Context 불일치 확인
Train 데이터 title-context 불일치 의심: 269개
Validation 데이터 title-context 불일치 의심: 17개

Train 불일치 의심 샘플 (최대 10개):

[1] ID: mrc-0-000748
    Title: '터미널 (2004년 영화)'
    공통 단어 비율: 0.00%
    Context 미리보기: 동유럽의 소국 크라코지아에서 온 빅토르 나보르스키가 미국 뉴욕의 존 F. 케네디 국제공항에 도착한다. 입국 심사대에 선 그는 여권이 유효하지 않다는 통보를 받는다. 알고보니 빅토르가 미국행 비행기를 타는 사이 크라코지아에서 쿠데타와 내전이 일어나 일시적인 유령 국가가 되었고, 빅토르는 미국에 입국할 수도, 고국에 돌아갈 수도 없는 신세가 된 것이다. 국장 ...

[2] ID: mrc-0-002011
    Title: '제1차 세계 대전'
    공통 단어 비율: 0.00%
    Context 미리보기: 양측 모두 2년 동안 서로에게 결정적인 타격을 줄 수 있는 공격을 하지 못했다. 1915~1917년 동안, 대영제국 및 프랑스는 전략, 전술적 방향의 측면의 선택 때문에 독일보다 더 많은 사상자로 고통받았다. 독일은 오직 하나의 주요 공세만 시도했지만 연합군은 독일의 방어선을 돌파하기 위한 여러 시도를 하였다.\n\n1916년 2월 독일은 프랑스의 베르됭에...

[3] ID: mrc-0-000250
    Title: '애순핑크 크릭 전투'
    공통 단어 비율: 0.00%
    Context 미리보기: 트렌턴에서 워싱턴은 난제에 부딪쳐 있었다. 소수를 제외하고는 모든 병력의 징병 기간이 12월 31일로 끝나기 때문에, 사병들을 설득해 징병 기간 연장을 승인받지 않는 한 군대가 싸우지 않고 붕괴될 것이라는 것을 알고 있었다. 따라서 30일에, 병사들에게 10달러의 상금과 1개월 연장 근무를 요청했다. 또한 무료로 지원병을 모집했지만 아무도 응하지 않았다. ...

[

## 14. HTML 태그/Markup 포함 여부 확인


In [32]:
# HTML 태그나 markup 포함 여부 확인
import re

def check_html_markup(df, dataset_name):
    html_pattern = re.compile(r'<[^>]+>')
    markup_pattern = re.compile(r'&[a-z]+;|&#\d+;')
    
    markup_indices = []
    markup_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        context = str(row['context'])
        
        html_tags = html_pattern.findall(context)
        markup_entities = markup_pattern.findall(context)
        
        if html_tags or markup_entities:
            markup_indices.append(idx)
            markup_details.append({
                'id': row['id'],
                'html_tags': len(html_tags),
                'markup_entities': len(markup_entities),
                'sample_tags': html_tags[:5] if html_tags else [],
                'sample_entities': markup_entities[:5] if markup_entities else []
            })
    
    return markup_indices, markup_details

train_markup, train_markup_details = check_html_markup(train_df, "Train")
val_markup, val_markup_details = check_html_markup(val_df, "Validation")

print("=" * 50)
print("14. HTML 태그/Markup 포함 여부 확인")
print("=" * 50)
print(f"Train 데이터 markup 포함: {len(train_markup)}개")
print(f"Validation 데이터 markup 포함: {len(val_markup)}개")

if len(train_markup_details) > 0:
    print(f"\nTrain markup 포함 샘플 (최대 10개):")
    for i, detail in enumerate(train_markup_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    HTML 태그 개수: {detail['html_tags']}")
        print(f"    Markup 엔티티 개수: {detail['markup_entities']}")
        if detail['sample_tags']:
            print(f"    샘플 태그: {detail['sample_tags']}")
        if detail['sample_entities']:
            print(f"    샘플 엔티티: {detail['sample_entities']}")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 11389.83it/s]

14. HTML 태그/Markup 포함 여부 확인
Train 데이터 markup 포함: 91개
Validation 데이터 markup 포함: 3개

Train markup 포함 샘플 (최대 10개):

[1] ID: mrc-0-005458
    HTML 태그 개수: 4
    Markup 엔티티 개수: 0
    샘플 태그: ['<미들랜즈 투데이>', '<도그 쇼>', '<데니스 매카시의 위클리 에코>', '<미들랜즈 투데이>']

[2] ID: mrc-0-004214
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<채색화 기법>']

[3] ID: mrc-0-002418
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<알마게스트의 발췌본>']

[4] ID: mrc-0-001141
    HTML 태그 개수: 2
    Markup 엔티티 개수: 0
    샘플 태그: ['<낚시>', '<담배피우는 남자(폭포)>']

[5] ID: mrc-0-005356
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<채색화 기법>']

[6] ID: mrc-0-001319
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<물리학사>']

[7] ID: mrc-0-005052
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<복지국가를 말한다>']

[8] ID: mrc-0-001592
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<서증의 진정성립과 증명력>']

[9] ID: mrc-0-004160
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<알마게스트의 발췌본>']

[10] ID: mrc-0-001265
    HTML 태그 개수: 1
    M

## 15. Answer Text 앞뒤 공백 확인


In [30]:
# Answer text의 앞뒤 공백 확인
def check_answer_whitespace(df, dataset_name):
    whitespace_indices = []
    whitespace_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        
        if isinstance(answers, dict) and 'text' in answers:
            texts = answers['text']
            
            if isinstance(texts, list):
                for text in texts:
                    if text != text.strip():
                        whitespace_indices.append(idx)
                        whitespace_details.append({
                            'id': row['id'],
                            'answer_text': text,
                            'answer_text_stripped': text.strip(),
                            'leading_space': text != text.lstrip(),
                            'trailing_space': text != text.rstrip(),
                            'repr': repr(text)
                        })
                        break
    
    return whitespace_indices, whitespace_details

train_ans_ws, train_ans_ws_details = check_answer_whitespace(train_df, "Train")
val_ans_ws, val_ans_ws_details = check_answer_whitespace(val_df, "Validation")

print("=" * 50)
print("15. Answer Text 앞뒤 공백 확인")
print("=" * 50)
print(f"Train 데이터 answer 앞뒤 공백: {len(train_ans_ws)}개")
print(f"Validation 데이터 answer 앞뒤 공백: {len(val_ans_ws)}개")

if len(train_ans_ws_details) > 0:
    print(f"\nTrain answer 공백 샘플 (최대 10개):")
    for i, detail in enumerate(train_ans_ws_details[:10]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer text: {detail['repr']}")
        print(f"    앞 공백: {detail['leading_space']}, 뒤 공백: {detail['trailing_space']}")
        print(f"    Trimmed: '{detail['answer_text_stripped']}'")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 11635.36it/s]

15. Answer Text 앞뒤 공백 확인
Train 데이터 answer 앞뒤 공백: 0개
Validation 데이터 answer 앞뒤 공백: 0개


In [33]:
# 전체 데이터 품질 요약
print("=" * 50)
print("데이터 품질 점검 요약")
print("=" * 50)

summary = {
    'Train': {
        '총 개수': len(train_df),
        '중복 ID': len(train_duplicate_ids),
        '빈 answer': len(train_empty_answers),
        '빈/짧은 context': len(train_empty_context) + len(train_short_context),
        '음수 answer_start': len(train_neg_start),
        '범위 초과 answer_start': len(train_out_range),
        'Annotation 불일치': len(train_mismatch_idx),
        'Span 범위 초과': len(train_span_out),
        '정답 중복 등장': len(train_dup_ans),
        'Whitespace mismatch': len(train_ws_mismatch),
        '오프셋 오류': len(train_offset_mismatch),
        'Context 이상치': len(train_outliers),
        'Question 길이 이상': len(train_long_questions),
        '중복 (Q,C) 쌍': len(train_dup_pairs),
        'Title-Context 불일치': len(train_title_mismatch),
        'HTML/Markup 포함': len(train_markup),
        'Answer 앞뒤 공백': len(train_ans_ws)
    },
    'Validation': {
        '총 개수': len(val_df),
        '중복 ID': len(val_duplicate_ids),
        '빈 answer': len(val_empty_answers),
        '빈/짧은 context': len(val_empty_context) + len(val_short_context),
        '음수 answer_start': len(val_neg_start),
        '범위 초과 answer_start': len(val_out_range),
        'Annotation 불일치': len(val_mismatch_idx),
        'Span 범위 초과': len(val_span_out),
        '정답 중복 등장': len(val_dup_ans),
        'Whitespace mismatch': len(val_ws_mismatch),
        '오프셋 오류': len(val_offset_mismatch),
        'Context 이상치': len(val_outliers),
        'Question 길이 이상': len(val_long_questions),
        '중복 (Q,C) 쌍': len(val_dup_pairs),
        'Title-Context 불일치': len(val_title_mismatch),
        'HTML/Markup 포함': len(val_markup),
        'Answer 앞뒤 공백': len(val_ans_ws)
    }
}

summary_df = pd.DataFrame(summary)
print(summary_df)

# 문제가 있는 데이터 비율
print("\n" + "=" * 50)
print("문제 데이터 비율")
print("=" * 50)

train_total = len(train_df)
val_total = len(val_df)

print(f"\nTrain 데이터:")
print(f"  - 중복 ID 비율: {len(train_duplicate_ids)/train_total*100:.2f}%")
print(f"  - 빈 answer 비율: {len(train_empty_answers)/train_total*100:.2f}%")
print(f"  - Annotation 불일치 비율: {len(train_mismatch_idx)/train_total*100:.2f}%")
print(f"  - Span 범위 초과 비율: {len(train_span_out)/train_total*100:.2f}%")
print(f"  - Whitespace mismatch 비율: {len(train_ws_mismatch)/train_total*100:.2f}%")
print(f"  - Context 이상치 비율: {len(train_outliers)/train_total*100:.2f}%")
print(f"  - 중복 (Q,C) 쌍 비율: {len(train_dup_pairs)/train_total*100:.2f}%")

print(f"\nValidation 데이터:")
print(f"  - 중복 ID 비율: {len(val_duplicate_ids)/val_total*100:.2f}%")
print(f"  - 빈 answer 비율: {len(val_empty_answers)/val_total*100:.2f}%")
print(f"  - Annotation 불일치 비율: {len(val_mismatch_idx)/val_total*100:.2f}%")
print(f"  - Span 범위 초과 비율: {len(val_span_out)/val_total*100:.2f}%")
print(f"  - Whitespace mismatch 비율: {len(val_ws_mismatch)/val_total*100:.2f}%")
print(f"  - Context 이상치 비율: {len(val_outliers)/val_total*100:.2f}%")
print(f"  - 중복 (Q,C) 쌍 비율: {len(val_dup_pairs)/val_total*100:.2f}%")


데이터 품질 점검 요약
                     Train  Validation
총 개수                  3952         240
중복 ID                    0           0
빈 answer                 0           0
빈/짧은 context             0           0
음수 answer_start          0           0
범위 초과 answer_start       0           0
Annotation 불일치           0           0
Span 범위 초과               0           0
정답 중복 등장              1312          78
Whitespace mismatch      0           0
오프셋 오류                   0           0
Context 이상치            155           7
Question 길이 이상           0           0
중복 (Q,C) 쌍               0           0
Title-Context 불일치      269          17
HTML/Markup 포함          91           3
Answer 앞뒤 공백             0           0

문제 데이터 비율

Train 데이터:
  - 중복 ID 비율: 0.00%
  - 빈 answer 비율: 0.00%
  - Annotation 불일치 비율: 0.00%
  - Span 범위 초과 비율: 0.00%
  - Whitespace mismatch 비율: 0.00%
  - Context 이상치 비율: 3.92%
  - 중복 (Q,C) 쌍 비율: 0.00%

Validation 데이터:
  - 중복 ID 비율: 0.00%
  - 빈 answer 비율: 0.00%
  - Annotation 불일치 비